# 🚦 Road-side video → textUpload a video, get back **where** each piece of text is and **what** it says,frame by frame — plus one entry per signboard instead of one per frame.Uses pretrained detection + recognition weights. **Nothing to train.**---### Before you start: turn the GPU on**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**On CPU this runs at ~2 fps. On a T4, ~20-30 fps. Do it now — switching laterrestarts the runtime and you lose everything below.Then: **Runtime → Run all**, and upload your video when cell 4 asks.

## 1 · Check the GPU

In [ ]:
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode == 0 and out.stdout.strip():
    print("GPU:", out.stdout.strip())
else:
    print("NO GPU  ->  Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("(It will still work on CPU, just ~10x slower.)")

## 2 · Install~1 minute. Colab already has torch and opencv, so this is mostly EasyOCR itself.Ignore any dependency-resolver warnings — they do not affect this notebook.

In [ ]:
%pip install -q easyocr
print("done")

## 3 · Write the codeFour modules. Nothing to clone — these cells create the files in `/content`.| file | what it does ||---|---|| `engines.py` | stage 1 detect + stage 2 recognise, and the crop between them || `tracker.py` | link boxes across frames, vote on the reading || `viz.py` | draw the annotated video || `detect_text_video.py` | the script that runs it all || `diagnose.py` | tries several settings on your video and says which works |

In [ ]:
%%writefile engines.py
"""Detection and recognition engines.

Two modes, and the difference is measurable -- on a 1280x720 clip with four
small tilted signs:

    mode=pipeline    3/4 read exactly, confidences 0.95-1.00
    mode=two_stage   2/4 read exactly, confidences 0.52-1.00

``pipeline`` hands the frame to the engine's own end-to-end call.  ``two_stage``
splits detection from recognition explicitly -- crop each detected polygon, read
each crop alone -- which is easier to reason about and worse at reading, because
it throws away the merging and contrast-retry logic the engine does between its
own two stages.  Use ``two_stage`` to see *which* stage is failing, then switch
back to ``pipeline`` for results.

Both modes expose ``detect()`` on its own, so stage-1 boxes can be drawn without
running recognition at all.

Engines ship pretrained weights that download on first use.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import cv2
import numpy as np


@dataclass
class Prediction:
    """One detected-and-read piece of text in one frame."""
    poly: np.ndarray      # (N, 2) float32
    text: str
    confidence: float

    @property
    def box(self) -> tuple:
        """Axis-aligned bounds as (x1, y1, x2, y2)."""
        x, y = self.poly[:, 0], self.poly[:, 1]
        return float(x.min()), float(y.min()), float(x.max()), float(y.max())


@dataclass
class DetectParams:
    """Knobs that decide what counts as text.  Defaults match the engine's own.

    ``mag`` is the one to reach for first on road video: the detector resizes
    the frame by this factor before looking, so 2.0 gives distant signage twice
    the pixels to be found in.  It costs roughly the square in time.

    ``link`` is the one behind "it only found half the words".  The detector
    finds character regions and links neighbours into a word; a high threshold
    links less, so "SPEED 40" comes back as "SPEED" and "40".  Lower it, and
    raise ``width_ths``, to merge more.
    """
    mag: float = 1.0              # pre-detection upscale
    low_text: float = 0.4         # lower  -> fainter strokes count as text
    text_threshold: float = 0.7   # lower  -> weaker regions count as text
    link: float = 0.4             # lower  -> neighbouring words merge
    width_ths: float = 0.5        # higher -> merge boxes further apart
    min_size: int = 10            # ignore boxes smaller than this (px)
    decoder: str = "greedy"       # or 'beamsearch' -- slower, a little better
    merge_words: bool = False     # group boxes into blocks; see below


# ``merge_words`` is what recovers a phrase the detector split.  On a frame
# whose signs read PHARMACY / EXIT 24 / MAIN STREET / SPEED 40, no combination
# of link and width_ths merged the last one -- it came back as "SPEED" and "40"
# every time.  Grouping does merge it, and reads all four exactly.
#
# It costs two things, so it is opt-in rather than the default:
#
# 1. Grouping discards per-box confidence, so every reading is scored 1.0.
#    Track agreement then counts frames that read a track the same way, rather
#    than weighting them by confidence, and --min-conf stops filtering.
# 2. It over-merges.  Grouping is geometric, so two *different* signs that drift
#    close together are joined as readily as two halves of one phrase -- on the
#    same clip, "EXIT 24" and "PHARMACY" became one track reading
#    "EXIT 24 PHARMACY" once they overlapped.
#
# So: turn it on when phrases arrive split, and check the tracks table for
# merges that should not be there.


# --------------------------------------------------------------------------
# cropping (two_stage mode only)
# --------------------------------------------------------------------------

def crop_polygon(image: np.ndarray, poly: np.ndarray, target_height: int = 64,
                 pad_y: float = 0.15, pad_x: float = 0.02) -> Optional[np.ndarray]:
    """Warp a (possibly rotated) quad to an upright crop for the recogniser.

    Padding is a fraction of the text *height* in both directions, and is much
    smaller horizontally than vertically.  Scaling the quad uniformly instead --
    the obvious implementation -- pads a long word by a lot horizontally and
    almost nothing vertically, which is backwards: recognisers want vertical
    margin, and the horizontal overshoot drags in whatever frames the text.  On
    a bordered sign that showed up as ``[PHARMACY]``: the panel edge, read as
    brackets.

    Returns None when the quad is degenerate.
    """
    poly = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if poly.shape[0] != 4:
        poly = cv2.boxPoints(cv2.minAreaRect(poly)).astype(np.float32)
    poly = _order_quad(poly)

    width = max(np.linalg.norm(poly[1] - poly[0]), np.linalg.norm(poly[2] - poly[3]))
    height = max(np.linalg.norm(poly[3] - poly[0]), np.linalg.norm(poly[2] - poly[1]))
    if width < 2 or height < 2:
        return None

    # Expand along the quad's own axes, so a tilted box stays tilted.
    x_axis = (poly[1] - poly[0]) / max(np.linalg.norm(poly[1] - poly[0]), 1e-6)
    y_axis = (poly[3] - poly[0]) / max(np.linalg.norm(poly[3] - poly[0]), 1e-6)
    dx, dy = x_axis * height * pad_x, y_axis * height * pad_y
    poly = np.array([poly[0] - dx - dy, poly[1] + dx - dy,
                     poly[2] + dx + dy, poly[3] - dx + dy], dtype=np.float32)

    out_h = int(target_height)
    out_w = max(int(round((width + 2 * height * pad_x) / (height + 2 * height * pad_y)
                          * out_h)), 8)
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]],
                   dtype=np.float32)
    return cv2.warpPerspective(image, cv2.getPerspectiveTransform(poly, dst),
                               (out_w, out_h), flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_REPLICATE)


def _order_quad(poly: np.ndarray) -> np.ndarray:
    """Order 4 points clockwise from the top-left corner."""
    s = poly.sum(axis=1)
    d = np.diff(poly, axis=1).ravel()
    return np.array([poly[np.argmin(s)], poly[np.argmin(d)],
                     poly[np.argmax(s)], poly[np.argmax(d)]], dtype=np.float32)


# --------------------------------------------------------------------------
# EasyOCR
# --------------------------------------------------------------------------

class EasyOCREngine:
    """CRAFT detector + CRNN recogniser (pure PyTorch, easiest to install)."""

    name = "easyocr"

    def __init__(self, langs: Sequence[str] = ("en",), gpu: bool = True,
                 params: Optional[DetectParams] = None):
        import easyocr  # lazy: --engine paddleocr should not need easyocr

        self.reader = easyocr.Reader(list(langs), gpu=gpu, verbose=False)
        self.p = params or DetectParams()

    @staticmethod
    def _rgb(image: np.ndarray) -> np.ndarray:
        """EasyOCR reads files as RGB but takes arrays as BGR, and the two give
        different results.  Feed it RGB so a frame matches the same frame saved
        to disk and passed by path."""
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def _detect_kwargs(self) -> dict:
        p = self.p
        return dict(text_threshold=p.text_threshold, low_text=p.low_text,
                    link_threshold=p.link, mag_ratio=p.mag, min_size=p.min_size,
                    width_ths=p.width_ths)

    def detect(self, image: np.ndarray) -> List[np.ndarray]:
        horizontal, free = self.reader.detect(self._rgb(image), **self._detect_kwargs())
        horizontal = horizontal[0] if horizontal else []
        free = free[0] if free else []

        polys: List[np.ndarray] = []
        for box in horizontal:
            # EasyOCR's horizontal boxes are (x_min, x_max, y_min, y_max) --
            # note the ordering, it is not the usual (x1, y1, x2, y2).
            x1, x2, y1, y2 = [float(v) for v in box[:4]]
            polys.append(np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]],
                                  dtype=np.float32))
        for quad in free:
            polys.append(np.asarray(quad, dtype=np.float32).reshape(-1, 2))
        return polys

    def read(self, image: np.ndarray) -> List[Prediction]:
        """End-to-end: detection and recognition in the engine's own pipeline."""
        results = self.reader.readtext(
            self._rgb(image), decoder=self.p.decoder,
            paragraph=self.p.merge_words, **self._detect_kwargs())
        out = []
        for item in results:
            # Grouped results are (quad, text); ungrouped are (quad, text, conf).
            quad, text = item[0], item[1]
            conf = float(item[2]) if len(item) > 2 else 1.0
            text = str(text).strip()
            if text:
                out.append(Prediction(np.asarray(quad, dtype=np.float32).reshape(-1, 2),
                                      text, conf))
        return out

    def recognize(self, image: np.ndarray, polys: Sequence[np.ndarray]) -> List[Prediction]:
        """Read each detected polygon in isolation (two_stage mode)."""
        grey = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        out: List[Prediction] = []
        for poly in polys:
            crop = crop_polygon(grey, poly)
            if crop is None:
                continue
            try:
                # With no box lists, recognize() treats the whole crop as one line.
                result = self.reader.recognize(crop, detail=1, decoder=self.p.decoder)
            except Exception:
                continue
            if not result:
                continue
            _, text, conf = result[0]
            text = str(text).strip()
            if text:
                out.append(Prediction(np.asarray(poly, dtype=np.float32), text,
                                      float(conf)))
        return out


# --------------------------------------------------------------------------
# PaddleOCR
# --------------------------------------------------------------------------

class PaddleOCREngine:
    """DBNet detector + SVTR recogniser.  Often sharper on small scene text."""

    name = "paddleocr"

    def __init__(self, langs: Sequence[str] = ("en",), gpu: bool = True,
                 params: Optional[DetectParams] = None):
        from paddleocr import PaddleOCR  # lazy, see EasyOCREngine

        self.p = params or DetectParams()
        self.ocr = self._build(PaddleOCR, (langs or ["en"])[0], gpu)

    @staticmethod
    def _build(PaddleOCR, lang: str, gpu: bool):
        """PaddleOCR's constructor keywords changed across 2.x and 3.x, so try
        the argument sets newest-first and keep the first that constructs."""
        for kwargs in ({"lang": lang, "use_textline_orientation": True},
                       {"lang": lang, "use_angle_cls": True, "use_gpu": gpu,
                        "show_log": False},
                       {"lang": lang}):
            try:
                return PaddleOCR(**kwargs)
            except TypeError:
                continue
        raise RuntimeError("could not construct PaddleOCR with any known signature")

    def _predict(self, image: np.ndarray) -> List[dict]:
        """Return each page's result dict, across 2.x/3.x return shapes."""
        pages: List[dict] = []
        if hasattr(self.ocr, "predict"):
            for page in self.ocr.predict(image) or []:
                if isinstance(page, dict):
                    pages.append(page)
                else:
                    pages.append(getattr(page, "json", {}).get("res", {}))
            if pages:
                return pages
        result = self.ocr.ocr(image)
        for page in result or []:
            if isinstance(page, dict):
                pages.append(page)
            elif isinstance(page, list):
                polys, texts, scores = [], [], []
                for item in page:
                    try:
                        polys.append(item[0])
                        texts.append(item[1][0])
                        scores.append(float(item[1][1]))
                    except (IndexError, TypeError):
                        continue
                pages.append({"dt_polys": polys, "rec_texts": texts,
                              "rec_scores": scores})
        return pages

    def detect(self, image: np.ndarray) -> List[np.ndarray]:
        polys: List[np.ndarray] = []
        for page in self._predict(image):
            for quad in page.get("dt_polys", []) or []:
                arr = np.asarray(quad, dtype=np.float32).reshape(-1, 2)
                if arr.shape[0] >= 4:
                    polys.append(arr)
        return polys

    def read(self, image: np.ndarray) -> List[Prediction]:
        out: List[Prediction] = []
        for page in self._predict(image):
            quads = page.get("dt_polys", []) or []
            texts = page.get("rec_texts", []) or []
            scores = page.get("rec_scores", []) or []
            for quad, text, score in zip(quads, texts, scores):
                text = str(text).strip()
                if text:
                    out.append(Prediction(
                        np.asarray(quad, dtype=np.float32).reshape(-1, 2),
                        text, float(score)))
        return out

    def recognize(self, image: np.ndarray, polys: Sequence[np.ndarray]) -> List[Prediction]:
        out: List[Prediction] = []
        for poly in polys:
            crop = crop_polygon(image, poly, target_height=48)
            if crop is None:
                continue
            try:
                pages = self._predict(crop)
            except Exception:
                continue
            for page in pages:
                texts = page.get("rec_texts", []) or []
                scores = page.get("rec_scores", []) or []
                if texts:
                    text = str(texts[0]).strip()
                    if text:
                        out.append(Prediction(np.asarray(poly, dtype=np.float32),
                                              text, float(scores[0]) if scores else 1.0))
                    break
        return out


# --------------------------------------------------------------------------
# presets
# --------------------------------------------------------------------------

PRESETS: Dict[str, dict] = {
    "default": {},
    # Distant road signage: look harder, and merge words the detector split.
    "small-text": dict(mag=2.0, low_text=0.3, text_threshold=0.6, link=0.2,
                       width_ths=1.0, min_size=6),
    # Same, but glue split phrases back together ("SPEED" + "40" -> "SPEED 40").
    "small-text-merged": dict(mag=2.0, low_text=0.3, text_threshold=0.6, link=0.2,
                              width_ths=1.0, min_size=6, merge_words=True),
    # Same, pushed further -- slow, for when small-text still misses things.
    "tiny-text": dict(mag=3.0, low_text=0.25, text_threshold=0.5, link=0.1,
                      width_ths=1.5, min_size=4, decoder="beamsearch"),
    # Long clips: fewer, bigger regions only.
    "fast": dict(mag=1.0, low_text=0.5, text_threshold=0.8, min_size=20),
}

ENGINES = {"easyocr": EasyOCREngine, "paddleocr": PaddleOCREngine}


def build_engine(name: str, langs: Sequence[str], gpu: bool,
                 params: Optional[DetectParams] = None):
    try:
        cls = ENGINES[name]
    except KeyError:
        raise SystemExit(f"unknown engine {name!r}; choose from {sorted(ENGINES)}")
    try:
        return cls(langs=langs, gpu=gpu, params=params)
    except ImportError as exc:
        raise SystemExit(
            f"engine {name!r} needs a package that is not installed ({exc}).\n"
            f"  easyocr    ->  pip install easyocr\n"
            f"  paddleocr  ->  pip install paddlepaddle paddleocr"
        ) from exc

In [ ]:
%%writefile tracker.py
"""Link per-frame detections into tracks, and vote on the transcription.

Per-frame OCR alone gives you the same signboard as 40 unrelated results, each
read slightly differently.  Two cheap steps fix that:

* **IoU association** -- a box that overlaps last frame's box is the same sign,
  so it keeps its id.  A short `max_age` keeps the id alive across a few frames
  of occlusion or missed detection.
* **Confidence voting** -- across every frame a track was seen, the reading with
  the highest summed confidence wins.  A word misread in three frames and read
  correctly in twenty comes out correct, which no single frame guarantees.

This is a deliberately simple appearance-free tracker: it uses geometry only.
It is enough for road-side video, where text moves smoothly and is rarely
duplicated within one frame.
"""

from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Sequence

import numpy as np


def iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection over union of two (x1, y1, x2, y2) boxes."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(ix2 - ix1, 0.0), max(iy2 - iy1, 0.0)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(a[2] - a[0], 0.0) * max(a[3] - a[1], 0.0)
    area_b = max(b[2] - b[0], 0.0) * max(b[3] - b[1], 0.0)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


@dataclass
class Track:
    track_id: int
    box: tuple
    poly: np.ndarray
    last_frame: int
    start_frame: int
    frames: Dict[int, List[List[float]]] = field(default_factory=dict)
    votes: Dict[str, float] = field(default_factory=lambda: defaultdict(float))
    hits: int = 0

    @property
    def text(self) -> str:
        """The reading with the highest total confidence over the track."""
        return max(self.votes.items(), key=lambda kv: kv[1])[0] if self.votes else ""

    @property
    def confidence(self) -> float:
        """Share of the track's total confidence that the winning reading holds.

        1.0 means every frame agreed; 0.4 means the reading is contested and the
        transcription should not be trusted much.
        """
        total = sum(self.votes.values())
        return (max(self.votes.values()) / total) if total > 0 else 0.0

    def to_dict(self) -> dict:
        return {
            "track_id": self.track_id,
            "text": self.text,
            "text_confidence": round(self.confidence, 4),
            "start_frame": self.start_frame,
            "end_frame": self.last_frame,
            "num_frames": len(self.frames),
            "frames": {str(k): v for k, v in sorted(self.frames.items())},
        }


class TextTracker:
    """Greedy IoU tracker over per-frame text predictions."""

    def __init__(self, iou_threshold: float = 0.3, max_age: int = 12,
                 min_hits: int = 2):
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks: List[Track] = []
        self._next_id = 1

    def update(self, predictions, frame_index: int) -> List[Track]:
        """Assign ids to this frame's predictions; returns the matched tracks."""
        live = [t for t in self.tracks if frame_index - t.last_frame <= self.max_age]

        pairs = sorted(
            ((iou(p.box, t.box), pi, ti)
             for pi, p in enumerate(predictions)
             for ti, t in enumerate(live)),
            key=lambda x: -x[0],
        )

        used_p, used_t, assigned = set(), set(), {}
        for score, pi, ti in pairs:
            if score < self.iou_threshold:
                break
            if pi in used_p or ti in used_t:
                continue
            used_p.add(pi)
            used_t.add(ti)
            assigned[pi] = live[ti]

        touched: List[Track] = []
        for pi, pred in enumerate(predictions):
            track = assigned.get(pi)
            if track is None:
                track = Track(track_id=self._next_id, box=pred.box, poly=pred.poly,
                              last_frame=frame_index, start_frame=frame_index)
                self._next_id += 1
                self.tracks.append(track)
            track.box = pred.box
            track.poly = pred.poly
            track.last_frame = frame_index
            track.hits += 1
            track.frames[frame_index] = np.asarray(pred.poly).round(1).tolist()
            track.votes[pred.text] += float(pred.confidence)
            touched.append(track)
        return touched

    def finished(self) -> List[Track]:
        """Tracks worth reporting, oldest first.

        Tracks seen fewer than ``min_hits`` times are dropped -- a box that
        appears in one frame and never again is almost always a false positive
        on a texture, not a real sign.
        """
        keep = [t for t in self.tracks if t.hits >= self.min_hits]
        return sorted(keep, key=lambda t: t.start_frame)

In [ ]:
%%writefile viz.py
"""Drawing helpers for the annotated output video."""

from __future__ import annotations

from typing import Optional, Sequence

import cv2
import numpy as np

# Distinguishable on road scenes (BGR).  Track id picks one by modulo, so the
# same sign keeps the same colour for as long as it holds its id -- which makes
# an id switch visible at a glance instead of buried in the JSON.
PALETTE = [
    (66, 133, 244), (52, 168, 83), (251, 188, 5), (234, 67, 53),
    (171, 71, 188), (0, 172, 193), (255, 112, 67), (158, 157, 36),
]


def colour_for(track_id: Optional[int]) -> tuple:
    if track_id is None:
        return (0, 215, 255)          # amber: detected but not yet tracked
    return PALETTE[track_id % len(PALETTE)]


def draw_polygon(frame: np.ndarray, poly: np.ndarray, colour: tuple,
                 thickness: int = 2) -> None:
    pts = np.asarray(poly, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(frame, [pts], isClosed=True, color=colour, thickness=thickness,
                  lineType=cv2.LINE_AA)


def draw_label(frame: np.ndarray, poly: np.ndarray, label: str, colour: tuple,
               scale: float = 0.55) -> None:
    """Filled caption above the polygon, nudged inside the frame if it overflows."""
    if not label:
        return
    pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    x, y = int(pts[:, 0].min()), int(pts[:, 1].min())

    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(label, font, scale, 1)
    pad = 4
    box_h = th + baseline + 2 * pad

    top = y - box_h
    if top < 0:                        # no room above -- put it below instead
        top = min(int(pts[:, 1].max()), frame.shape[0] - box_h - 1)
    top = max(top, 0)
    left = max(min(x, frame.shape[1] - tw - 2 * pad - 1), 0)

    cv2.rectangle(frame, (left, top), (left + tw + 2 * pad, top + box_h),
                  colour, thickness=-1)
    cv2.putText(frame, label, (left + pad, top + th + pad), font, scale,
                (255, 255, 255), 1, cv2.LINE_AA)


def annotate(frame: np.ndarray, items: Sequence[dict], show_conf: bool = True,
             stage: str = "both") -> np.ndarray:
    """Draw one frame's results.

    ``items`` are dicts with ``poly`` and optionally ``text``, ``confidence``
    and ``track_id``.  ``stage='detect'`` draws boxes only -- useful for seeing
    what detection did before recognition had a chance to spoil it.
    """
    out = frame.copy()
    for item in items:
        colour = colour_for(item.get("track_id"))
        draw_polygon(out, item["poly"], colour)
        if stage == "detect":
            continue
        text = item.get("text", "")
        if not text:
            continue
        parts = []
        if item.get("track_id") is not None:
            parts.append(f"#{item['track_id']}")
        parts.append(text)
        if show_conf and item.get("confidence") is not None:
            parts.append(f"{item['confidence']:.2f}")
        draw_label(out, item["poly"], " ".join(parts), colour)
    return out


def draw_hud(frame: np.ndarray, lines: Sequence[str]) -> None:
    """Small translucent status panel in the top-left corner."""
    if not lines:
        return
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, pad, gap = 0.5, 8, 20
    width = max(cv2.getTextSize(t, font, scale, 1)[0][0] for t in lines) + 2 * pad
    height = gap * len(lines) + pad

    panel = frame[0:height, 0:width]
    if panel.size:
        frame[0:height, 0:width] = cv2.addWeighted(
            panel, 0.35, np.zeros_like(panel), 0.0, 0)
    for i, text in enumerate(lines):
        cv2.putText(frame, text, (pad, pad + gap * i + 6), font, scale,
                    (255, 255, 255), 1, cv2.LINE_AA)

In [ ]:
%%writefile detect_text_video.py
#!/usr/bin/env python3
"""Read the text in a road-side video, frame by frame.

Two explicit stages per frame:

    stage 1  DETECT     where is the text?      -> polygons
    stage 2  RECOGNISE  what does it say?       -> transcription + confidence

then an optional third pass links detections across frames so one signboard is
reported once, with the reading its frames agreed on, instead of once per frame.

    python detect_text_video.py --video road.mp4 --render out.mp4 --out results.json

Speed: recognition dominates, and it runs once per detected box.  ``--every 3``
processes one frame in three (the output video still has every frame -- skipped
frames reuse the previous result), which is usually invisible at 25-30 fps and
roughly triples throughput.
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import cv2
import numpy as np

from engines import PRESETS, DetectParams, build_engine
from tracker import TextTracker
from viz import annotate, draw_hud


def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Detect and read text in a video, frame by frame.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)

    p.add_argument("--video", required=True, help="input video file")
    p.add_argument("--out", default=None, help="write results here (JSON)")
    p.add_argument("--render", default=None, help="write an annotated video here")
    p.add_argument("--frames-dir", default=None,
                   help="also save annotated frames as JPGs in this directory")

    p.add_argument("--engine", default="easyocr", choices=["easyocr", "paddleocr"])
    p.add_argument("--langs", default="en",
                   help="comma-separated language codes, e.g. en or en,hi")
    p.add_argument("--cpu", action="store_true", help="force CPU even if a GPU is present")
    p.add_argument("--mode", default="pipeline", choices=["pipeline", "two_stage"],
                   help="'pipeline' is the engine's own end-to-end call and reads "
                        "better; 'two_stage' crops each detection and reads it "
                        "alone, which is worse but shows which stage failed")

    p.add_argument("--preset", default="default", choices=sorted(PRESETS),
                   help="detector tuning; 'small-text' for distant road signage")
    p.add_argument("--mag", type=float, default=None,
                   help="upscale by this before detection (the knob for small text)")
    p.add_argument("--low-text", type=float, default=None,
                   help="lower = fainter strokes count as text")
    p.add_argument("--text-threshold", type=float, default=None,
                   help="lower = weaker regions count as text")
    p.add_argument("--link", type=float, default=None,
                   help="lower = neighbouring words merge into one box")
    p.add_argument("--width-ths", type=float, default=None,
                   help="higher = merge boxes that are further apart")
    p.add_argument("--min-size", type=int, default=None,
                   help="ignore detections smaller than this many pixels")
    p.add_argument("--decoder", default=None, choices=["greedy", "beamsearch"],
                   help="beamsearch is slower and a little more accurate")
    p.add_argument("--merge-words", dest="merge_words", action="store_true",
                   default=None,
                   help="glue phrases the detector split ('SPEED'+'40' -> "
                        "'SPEED 40'); costs per-box confidence, so --min-conf "
                        "stops filtering and agreement counts frames instead")

    p.add_argument("--every", type=int, default=1,
                   help="run OCR on every Nth frame (1 = every frame)")
    p.add_argument("--max-frames", type=int, default=0,
                   help="stop after this many frames (0 = whole video)")
    p.add_argument("--resize", type=int, default=0,
                   help="resize the long side to this before OCR (0 = native). "
                        "Downscaling is a speed knob and it costs you small text")

    p.add_argument("--min-conf", type=float, default=0.30,
                   help="drop readings below this confidence")
    p.add_argument("--min-chars", type=int, default=2,
                   help="drop readings shorter than this many characters")

    p.add_argument("--no-track", action="store_true",
                   help="report every frame separately, no ids, no voting")
    p.add_argument("--iou", type=float, default=0.30, help="tracker IoU threshold")
    p.add_argument("--max-age", type=int, default=12,
                   help="frames a track survives without a detection")
    p.add_argument("--min-hits", type=int, default=2,
                   help="a track needs this many frames to be reported")

    p.add_argument("--stage", default="both", choices=["both", "detect"],
                   help="'detect' draws stage-1 boxes only, skipping recognition")
    return p.parse_args(argv)


def open_video(path: str):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise SystemExit(f"cannot open video: {path}")
    meta = {
        "fps": cap.get(cv2.CAP_PROP_FPS) or 25.0,
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    return cap, meta


def scale_for(frame: np.ndarray, long_side: int) -> float:
    """Downscale factor so OCR runs on a smaller frame; 1.0 means leave it alone."""
    if long_side <= 0:
        return 1.0
    longest = max(frame.shape[:2])
    return (long_side / longest) if longest > long_side else 1.0


def main(argv=None) -> int:
    args = parse_args(argv)

    cap, meta = open_video(args.video)
    total = meta["frames"] if args.max_frames <= 0 else min(meta["frames"], args.max_frames)
    print(f"video   : {args.video}")
    print(f"          {meta['width']}x{meta['height']}  {meta['fps']:.1f} fps  "
          f"{meta['frames']} frames")

    params = DetectParams(**PRESETS[args.preset])
    for name in ("mag", "low_text", "text_threshold", "link", "width_ths",
                 "min_size", "decoder", "merge_words"):
        value = getattr(args, name)
        if value is not None:              # an explicit flag overrides the preset
            setattr(params, name, value)

    print(f"engine  : {args.engine}  mode={args.mode}  preset={args.preset}")
    print(f"          mag={params.mag} low_text={params.low_text} "
          f"link={params.link} min_size={params.min_size}")
    print("          (first run downloads pretrained weights)")
    engine = build_engine(args.engine, [s.strip() for s in args.langs.split(",")],
                          gpu=not args.cpu, params=params)

    # Nothing to track in detect-only mode: tracks are keyed on readings.
    track_off = args.no_track or args.stage == "detect"
    tracker = None if track_off else TextTracker(
        iou_threshold=args.iou, max_age=args.max_age, min_hits=args.min_hits)

    writer = None
    if args.render:
        Path(args.render).parent.mkdir(parents=True, exist_ok=True)
        writer = cv2.VideoWriter(args.render, cv2.VideoWriter_fourcc(*"mp4v"),
                                 meta["fps"], (meta["width"], meta["height"]))
        if not writer.isOpened():
            raise SystemExit(f"cannot write video: {args.render}")

    frames_dir = None
    if args.frames_dir:
        frames_dir = Path(args.frames_dir)
        frames_dir.mkdir(parents=True, exist_ok=True)

    per_frame: list = []
    last_items: list = []          # reused for frames we skip, so video stays smooth
    n_detected = n_read = processed = 0
    frame_index = -1
    started = time.time()

    while True:
        if args.max_frames > 0 and processed >= args.max_frames:
            break
        ok, frame = cap.read()
        if not ok:
            break
        frame_index += 1
        processed += 1

        if frame_index % args.every == 0:
            scale = scale_for(frame, args.resize)
            small = (cv2.resize(frame, None, fx=scale, fy=scale,
                                interpolation=cv2.INTER_AREA)
                     if scale != 1.0 else frame)

            if args.stage == "detect":
                # ---- stage 1 only: where is the text? -----------------------
                polys = engine.detect(small)
                preds = []
            elif args.mode == "pipeline":
                # The engine runs both stages itself and returns boxes + text.
                preds = engine.read(small)
                polys = [p.poly for p in preds]
            else:
                # ---- stage 1: where? ---- then ---- stage 2: what? ----------
                polys = engine.detect(small)
                preds = engine.recognize(small, polys)

            n_detected += len(polys)
            if preds:
                preds = [p for p in preds
                         if p.confidence >= args.min_conf
                         and len(p.text) >= args.min_chars]
                n_read += len(preds)

            # back to full-resolution coordinates
            if scale != 1.0:
                inv = 1.0 / scale
                for p in preds:
                    p.poly = p.poly * inv
                polys = [np.asarray(q, dtype=np.float32) * inv for q in polys]

            if args.stage == "detect":
                last_items = [{"poly": q, "track_id": None} for q in polys]
            elif tracker is None:
                last_items = [{"poly": p.poly, "text": p.text,
                               "confidence": p.confidence, "track_id": None}
                              for p in preds]
            else:
                tracks = tracker.update(preds, frame_index)
                last_items = [{"poly": p.poly, "text": t.text,
                               "confidence": p.confidence, "track_id": t.track_id}
                              for p, t in zip(preds, tracks)]

            per_frame.append({
                "frame": frame_index,
                "detections": [
                    {"poly": np.asarray(i["poly"]).round(1).tolist(),
                     "text": i.get("text", ""),
                     "confidence": round(float(i.get("confidence") or 0.0), 4),
                     "track_id": i.get("track_id")}
                    for i in last_items
                ],
            })

        if writer is not None or frames_dir is not None:
            shown = annotate(frame, last_items, stage=args.stage)
            draw_hud(shown, [f"frame {frame_index}",
                             f"text in view: {len(last_items)}"])
            if writer is not None:
                writer.write(shown)
            if frames_dir is not None:
                cv2.imwrite(str(frames_dir / f"frame_{frame_index:06d}.jpg"), shown)

        if total > 0 and processed % 25 == 0:
            elapsed = time.time() - started
            fps = processed / elapsed if elapsed > 0 else 0.0
            eta = (total - processed) / fps if fps > 0 else 0.0
            sys.stdout.write(f"\r  {processed}/{total} frames  {fps:5.1f} fps  "
                             f"ETA {eta/60:4.1f} min ")
            sys.stdout.flush()

    cap.release()
    if writer is not None:
        writer.release()
    print()

    elapsed = time.time() - started
    print(f"processed {processed} frames in {elapsed:.1f}s "
          f"({processed / max(elapsed, 1e-6):.1f} fps)")
    print(f"stage 1  : {n_detected} text regions detected")
    print(f"stage 2  : {n_read} readings kept (conf >= {args.min_conf})")

    results = {
        "video": str(args.video),
        "engine": args.engine,
        "meta": meta,
        "settings": {"every": args.every, "resize": args.resize,
                     "min_conf": args.min_conf, "tracking": tracker is not None},
        "frames": per_frame,
    }
    if tracker is not None:
        tracks = tracker.finished()
        results["tracks"] = [t.to_dict() for t in tracks]
        print(f"tracking : {len(tracks)} distinct pieces of text")
        for t in tracks[:25]:
            print(f"           #{t.track_id:<3} {t.text!r:<28} "
                  f"frames {t.start_frame}-{t.last_frame}  "
                  f"agreement {t.confidence:.2f}")
        if len(tracks) > 25:
            print(f"           ... and {len(tracks) - 25} more")

    if args.out:
        Path(args.out).parent.mkdir(parents=True, exist_ok=True)
        Path(args.out).write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"wrote    : {args.out}")
    if args.render:
        print(f"wrote    : {args.render}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile diagnose.py
#!/usr/bin/env python3
"""Find settings that work on *your* video, by trying them and counting.

Reading nothing has a small number of causes and they need opposite fixes, so
guessing wastes runs.  This samples a few frames, tries each setting on all of
them, and reports what each found:

    python diagnose.py --video road.mp4

    setting        regions  read  mean conf  sample
    -------------------------------------------------------------
    default              6     2       0.41  'PHARMACY', 'Bu2'
    small-text          14    11       0.78  'PHARMACY', 'EXIT 24', 'MAIN ST'
    tiny-text           17    12       0.71  'PHARMACY', 'EXIT 24', 'MAIN ST'
    two_stage/small     14     8       0.55  '[PHARMACY]', 'EXIT 24'

Take the row with the most *read* at a decent mean confidence and pass its
setting to detect_text_video.py.  `--dump` writes each row's boxes as an image
so you can see what it found rather than trusting the count.

Reading the rows:

* **regions high, read low** -- detection is fine, recognition is not.  The
  text is too blurred or too small to resolve; try `--mag 3`, or accept that a
  frame that is unreadable to you is unreadable to the model too.
* **regions low everywhere** -- detection is failing.  The text is smaller than
  `min_size`, or fainter than `low_text`.  `tiny-text` addresses both.
* **words arriving split** ("SPEED", "40" instead of "SPEED 40") -- lower
  `link` and raise `width_ths`; `small-text` already does.
* **nothing works at any setting** -- crop to the region that matters, or use a
  higher-resolution source.  No setting recovers detail the video does not have.
"""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import List

import cv2
import numpy as np

from engines import PRESETS, DetectParams, build_engine


def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Try several settings on a few frames and report what each found.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--video", required=True)
    p.add_argument("--frames", type=int, default=3, help="how many frames to sample")
    p.add_argument("--engine", default="easyocr", choices=["easyocr", "paddleocr"])
    p.add_argument("--langs", default="en")
    p.add_argument("--cpu", action="store_true")
    p.add_argument("--dump", default=None,
                   help="directory to write one annotated image per setting")
    p.add_argument("--min-conf", type=float, default=0.3)
    return p.parse_args(argv)


def sample_frames(path: str, count: int) -> List[np.ndarray]:
    """Frames spread across the clip, skipping the very start and end."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise SystemExit(f"cannot open video: {path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for i in range(count):
        pos = int(total * (i + 1) / (count + 1)) if total > 0 else i
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ok, frame = cap.read()
        if ok:
            frames.append(frame)
    cap.release()
    if not frames:
        raise SystemExit("could not read any frames")
    return frames


def main(argv=None) -> int:
    args = parse_args(argv)
    frames = sample_frames(args.video, args.frames)
    h, w = frames[0].shape[:2]
    print(f"video   : {args.video}  ({w}x{h}, sampled {len(frames)} frames)")
    print(f"engine  : {args.engine}\n")

    langs = [s.strip() for s in args.langs.split(",")]
    trials = [(name, name, "pipeline") for name in
              ("default", "small-text", "tiny-text")]
    trials.append(("two_stage/small-text", "small-text", "two_stage"))

    dump_dir = Path(args.dump) if args.dump else None
    if dump_dir:
        dump_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for label, preset, mode in trials:
        params = DetectParams(**PRESETS[preset])
        engine = build_engine(args.engine, langs, gpu=not args.cpu, params=params)

        regions = reads = 0
        confs: List[float] = []
        texts: List[str] = []
        for i, frame in enumerate(frames):
            if mode == "pipeline":
                preds = engine.read(frame)
                polys = [p.poly for p in preds]
            else:
                polys = engine.detect(frame)
                preds = engine.recognize(frame, polys)
            kept = [p for p in preds if p.confidence >= args.min_conf]
            regions += len(polys)
            reads += len(kept)
            confs.extend(p.confidence for p in kept)
            texts.extend(p.text for p in kept)

            if dump_dir and i == 0:
                from viz import annotate
                items = [{"poly": p.poly, "text": p.text,
                          "confidence": p.confidence, "track_id": None}
                         for p in kept]
                name = label.replace("/", "_")
                cv2.imwrite(str(dump_dir / f"{name}.jpg"), annotate(frame, items))

        mean_conf = float(np.mean(confs)) if confs else 0.0
        rows.append((label, regions, reads, mean_conf, texts))

    print(f"{'setting':<22}{'regions':>8}{'read':>6}{'mean conf':>11}   sample")
    print("-" * 92)
    best = max(rows, key=lambda r: (r[2], r[3]))
    for label, regions, reads, mean_conf, texts in rows:
        uniq = list(dict.fromkeys(texts))[:4]
        mark = "  <-- best" if (label, regions, reads) == best[:3] else ""
        print(f"{label:<22}{regions:>8}{reads:>6}{mean_conf:>11.2f}   "
              + ", ".join(repr(t) for t in uniq)[:44] + mark)

    print()
    label, regions, reads, mean_conf, _ = best
    if reads == 0:
        print("Nothing was read at any setting.  Detection found "
              f"{max(r[1] for r in rows)} regions at best, so:")
        print("  - regions > 0 : the text is detected but too degraded to resolve.")
        print("                  Try --mag 4, or a higher-resolution source.")
        print("  - regions = 0 : nothing looks like text at this scale.  Check the")
        print("                  frame is what you expect (--dump) and that the")
        print("                  text is not smaller than a few pixels tall.")
        return 1

    preset = label.split("/")[-1]
    mode = "two_stage" if label.startswith("two_stage") else "pipeline"
    print(f"Best: {label} -- {reads} readings at mean confidence {mean_conf:.2f}\n")
    print("Run the video with:")
    print(f"  python detect_text_video.py --video {args.video} \\")
    print(f"      --preset {preset} --mode {mode} --render annotated.mp4 --out results.json")
    if mean_conf < 0.5:
        print("\nMean confidence is low -- expect wrong readings.  Trajectory voting")
        print("recovers some of it, so judge the tracks table rather than a frame.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 4 · Give it a videoRun the cell and pick a file (mp4, avi, mov). **Keep the first try short —10-30 seconds** — so you see the whole pipeline work before spending minutes ona long clip.No video handy? Skip to the cell below it and a test clip is generated for you.

In [ ]:
from google.colab import files

VIDEO = None
uploaded = files.upload()
if uploaded:
    VIDEO = list(uploaded)[0]
    print("using:", VIDEO)
else:
    print("nothing uploaded -- run the next cell to generate a test clip instead")

### …or generate a test clip (only if you did not upload one)

In [ ]:
import cv2, numpy as np, os

if not VIDEO or not os.path.exists(VIDEO):
    W, H, N, FPS = 960, 540, 90, 25
    vw = cv2.VideoWriter("demo_road.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
    signs = [("PHARMACY", (30,120,220), 700, 150, 0.9, -4.5),
             ("EXIT 24",  (40,140,60),  520,  90, 0.7, -3.0),
             ("MAIN ST",  (200,140,40), 260, 300, 0.6, -2.0)]
    for f in range(N):
        img = np.zeros((H, W, 3), np.uint8)
        img[:H//2] = (170,150,130); img[H//2:] = (70,70,72)
        cv2.line(img, (W//2,H//2), (W//2-200,H), (220,220,220), 3)
        for x in range(0, W, 90):
            cv2.rectangle(img, (x,H//2-30), (x+40,H//2), (90,110,80), -1)
        for text, col, x0, y0, s, vx in signs:
            sc = s*(1+0.012*f); x = int(x0+vx*f); y = int(y0+0.6*f)
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, sc, 2)
            if x+tw+24 < 0 or x > W: continue
            cv2.rectangle(img, (x-12,y-th-12), (x+tw+12,y+12), (245,245,245), -1)
            cv2.rectangle(img, (x-12,y-th-12), (x+tw+12,y+12), col, 3)
            cv2.putText(img, text, (x,y), cv2.FONT_HERSHEY_SIMPLEX, sc, col, 2, cv2.LINE_AA)
        img = cv2.GaussianBlur(img, (3,3), 0)
        vw.write(img)
    vw.release()
    VIDEO = "demo_road.mp4"
    print("generated test clip:", VIDEO)
else:
    print("already have a video:", VIDEO)

## 5 · Which settings suit *your* video?Every video is different, and the failure causes need opposite fixes — someasure instead of guessing. This samples three frames, tries each setting, andtells you which found the most text.Takes a couple of minutes. Worth it before running the whole clip.

In [ ]:
!python diagnose.py --video "$VIDEO" --frames 3

## 6 · Run it`--every 2` reads one frame in two. The output video still has every frame, andat 25-30 fps the reuse is invisible — it just halves the work.Slower than you want? Raise `--every` to 3 or 4, or drop `--resize` to 960.Missing small distant signs? Raise `--resize` to 1920 and lower `--min-conf`.

In [ ]:
!python detect_text_video.py \
    --video "$VIDEO" \
    --out results.json \
    --render annotated.mp4 \
    --preset small-text \
    --every 2 \
    --min-conf 0.3

# Use whatever --preset cell 5 recommended.
# Phrases coming out split ("SPEED", "40")?  add --merge-words

## 7 · Look at the resultThree frames spread across the clip. Each box is one detection; the label is`#track_id TEXT confidence`, and the colour is keyed to the track id — so if asign changes colour mid-clip, it lost its id there.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

cap = cv2.VideoCapture("annotated.mp4")
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
picks = [int(total*r) for r in (0.15, 0.45, 0.75)]

fig, axes = plt.subplots(len(picks), 1, figsize=(13, 7*len(picks)))
for ax, idx in zip(np.atleast_1d(axes), picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"frame {idx}", fontsize=10)
    ax.axis("off")
cap.release()
plt.tight_layout(); plt.show()

### What it found

In [ ]:
import json
r = json.load(open("results.json"))

print(f"{len(r['frames'])} frames processed\n")
tracks = r.get("tracks", [])
print(f"{len(tracks)} distinct pieces of text:\n")
print(f"{'id':<5}{'text':<30}{'frames':<16}{'agreement'}")
print("-"*66)
for t in tracks:
    span = f"{t['start_frame']}-{t['end_frame']}"
    print(f"{t['track_id']:<5}{t['text'][:28]:<30}{span:<16}{t['text_confidence']:.2f}")

`agreement` is **not** the recogniser's score. It is the share of the track'stotal confidence held by the winning reading: `1.00` means every frame read itthe same way, `0.40` means the frames disagreed and you should not trust thattranscription.

## 8 · Play the annotated video hereColab's player will not play the `mp4v` file the script writes, so thisre-encodes to H.264 first. Skip it for long clips — the player embeds the wholefile in the page — and just download it in the next cell.

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

!ffmpeg -y -loglevel error -i annotated.mp4 -c:v libx264 -pix_fmt yuv420p annotated_h264.mp4

size_mb = os.path.getsize("annotated_h264.mp4") / 1e6
if size_mb > 40:
    print(f"{size_mb:.0f} MB is too big to embed -- download it in the next cell instead.")
else:
    data = b64encode(open("annotated_h264.mp4","rb").read()).decode()
    display(HTML(f'<video width=880 controls><source src="data:video/mp4;base64,{data}"></video>'))

## 9 · Download

In [ ]:
from google.colab import files
files.download("results.json")
files.download("annotated_h264.mp4" if os.path.exists("annotated_h264.mp4") else "annotated.mp4")

---## TuningRun the run-cell again with different flags — nothing above it needs re-running.| Problem | Fix ||---|---|| Too slow | `--every 4`, or `--resize 960` || Small/distant signs missed | `--preset tiny-text` || Phrases split ("SPEED", "40") | `--merge-words` (it can over-merge — check) || Junk readings off road texture | `--min-conf 0.5 --min-chars 3` || One sign gets several ids | `--max-age 25`, or `--iou 0.2` || Want per-frame output, no ids | `--no-track` || Hindi as well as English | `--langs en,hi` |**Is it detection or recognition that is failing?** Run with `--stage detect`.That draws stage-1 boxes and skips reading entirely. Box present but text wrong→ recognition, try `--mag 3`. No box at all → detection, try `--preset tiny-text`.---## What this isA baseline built on **off-the-shelf pretrained OCR** — someone else's weights,wrapped in a two-stage pipeline plus IoU tracking.It is **not** the `vtspot/` thesis system, which trains from randominitialisation and shares no code with this. Presenting this as a baseline tocompare against is honest and useful; presenting it as the thesis system is not.